# 01 — Explore NASA FIRMS data (Indonesia bbox, Aug–Oct 2019)

Real data: FIRMS standard-processing (SP) archive for MODIS (Terra+Aqua), VIIRS S-NPP, VIIRS NOAA-20.
Pulled with `eyes_on_karhutla.ingestion.fetch_area` over `config.INDONESIA_BBOX` (includes neighbouring countries; not yet clipped).

Hotspots are thermal-anomaly detections, not confirmed fires.

In [4]:
import pandas as pd
import plotly.express as px
from eyes_on_karhutla import ingestion, preprocessing

SOURCES = ['MODIS_SP', 'VIIRS_SNPP_SP', 'VIIRS_NOAA20_SP']
raw = pd.concat([ingestion.read_firms_csv(ingestion.latest_raw(s), s) for s in SOURCES], ignore_index=True)
obs, excluded = preprocessing.normalize(raw)
len(raw), len(obs), len(excluded)

(799948, 799948, 0)

## Columns and value domains per source

In [2]:
obs.groupby('source').agg(rows=('latitude', 'size'), satellites=('satellite', lambda s: sorted(s.unique())),
    first=('acq_datetime', 'min'), last=('acq_datetime', 'max'), days=('acq_date', 'nunique'))

,rows,satellites,first,last,days
source,,,,,
MODIS_SP,75483,"[Aqua, Terra]",2019-08-01 02:31:00+00:00,2019-10-31 19:11:00+00:00,92
VIIRS_NOAA20_SP,362218,[N20],2019-08-01 05:23:00+00:00,2019-10-31 18:04:00+00:00,92
VIIRS_SNPP_SP,362247,[N],2019-08-01 04:34:00+00:00,2019-10-31 18:55:00+00:00,92


In [3]:
pd.crosstab(obs['source'], obs['type'], margins=True)  # FIRMS type: 0 vegetation fire, 1 volcano, 2 static land, 3 offshore

type,0,1,2,3,All
source,,,,,
MODIS_SP,72951,845,1066,621,75483
VIIRS_NOAA20_SP,352181,1618,5642,2777,362218
VIIRS_SNPP_SP,355404,1448,3661,1734,362247
All,780536,3911,10369,5132,799948


In [4]:
viirs = obs[obs.sensor == 'VIIRS']
pd.crosstab(viirs['source'], viirs['confidence'], normalize='index').round(3)

confidence,h,l,n
source,,,
VIIRS_NOAA20_SP,0.039,0.07,0.892
VIIRS_SNPP_SP,0.042,0.07,0.888


In [5]:
modis = obs[obs.sensor == 'MODIS']
pd.to_numeric(modis['confidence']).describe()

count      75483.0
mean     67.414265
std      23.594078
min            0.0
25%           52.0
50%           70.0
75%           86.0
max          100.0
Name: confidence, dtype: Float64

## Pixel size and FRP differ by sensor (resolution effect)

In [6]:
obs.groupby('source')[['scan', 'track', 'frp']].describe().T.round(2)

source       MODIS_SP  VIIRS_NOAA20_SP  VIIRS_SNPP_SP
scan  count  75483.00        362218.00      362247.00
      mean       1.53             0.46           0.46
      std        0.74             0.08           0.08
      min        1.00             0.32           0.32
      25%        1.00             0.39           0.39
      50%        1.20             0.43           0.43
      75%        1.70             0.51           0.50
      max        4.80             0.80           0.80
track count  75483.00        362218.00      362247.00
      mean       1.19             0.47           0.47
      std        0.23             0.11           0.11
      min        1.00             0.36           0.36
      25%        1.00             0.38           0.38
      50%        1.10             0.44           0.44
      75%        1.30             0.55           0.55
      max        2.00             0.78           0.78
frp   count  75483.00        362218.00      362247.00
      mean      38.68             8.36           8.51
      std       95.62            17.18          17.07
      min        0.00             0.00           0.00
      25%       11.00             2.21           2.25
      50%       19.30             4.61           4.65
      75%       38.30             8.80           8.78
      max     6647.60          1840.60        1322.99

## Acquisition times (UTC) — overpass structure

In [7]:
obs['hour_utc'] = obs['acq_datetime'].dt.hour
hours = obs.groupby(['hour_utc', 'satellite']).size().rename('detections').reset_index()
px.bar(hours, x='hour_utc', y='detections', color='satellite', barmode='group',
       title='Detections by UTC hour (WIB = UTC+7)')

## Daily detection counts per source (raw, not harmonized)

Raw counts are not comparable across sensors: VIIRS 375 m pixels yield more detections per fire than MODIS 1 km.

In [8]:
daily = obs.groupby([obs['acq_datetime'].dt.date.rename('date'), 'source']).size().rename('detections').reset_index()
px.line(daily, x='date', y='detections', color='source', title='Daily detections (raw counts)')

In [9]:
wide = daily.pivot(index='date', columns='source', values='detections').fillna(0)
ratio = (wide['VIIRS_SNPP_SP'] / wide['MODIS_SP']).replace([float('inf')], pd.NA)
ratio.describe(), wide.corr().round(3)

(count    92.000000
 mean      5.307415
 std       1.627592
 min       1.066890
 25%       4.191792
 50%       5.045627
 75%       6.452647
 max      10.287770
 dtype: float64,
 source           MODIS_SP  VIIRS_NOAA20_SP  VIIRS_SNPP_SP
 source                                                   
 MODIS_SP            1.000            0.903          0.896
 VIIRS_NOAA20_SP     0.903            1.000          0.980
 VIIRS_SNPP_SP       0.896            0.980          1.000)

## Findings (to be recorded in lode/data)

- See lode/data/firms-sample-2019.md.